# Ni triple-axis exercise
This notebook contains both the widget to run the simulation, but also some Python code at the end, to show how one can plot the scan data in other ways, and get more information out of them using Python.

## Run the simulation
These two cells first load in the code for the instrument, and then opens up the widget for you to run the simulation with set parameters.

In [1]:
import code_folder.exercise as exercise

In [ ]:
%matplotlib widget
sim_interface = exercise.make_widget_interface()
sim_interface.show_interface()

## Scan a parameter
After selecting the other parameters above, you can now scan `QHL` (or other parameters, just change the code in the cell) around the chosen reciprocal-space position. The live plot shows the `He3H` monitor intensity for each scan point, and you can follow how far the scan has gone with the lines appearing below that.

In [ ]:
scan = exercise.scan(
    sim_interface,
    parameter_name="QL",   # change this parameter name to whichever parameter you want to scan over
    start=3.8,             # start ...
    stop=4.2,              # ... and end steps of the scan
    steps=21,              # number of simulation steps to do
    monitor_name="He3H",
    ncount=1e6,            # number of neutron rays to simulate for each scan step
)
scan.plot_monitor("He3H")

### Some other ways to manipulate the scan data
The following cells are Python code examples on how to plot the scan data in a log scale, and how to calculate things with it.

#### Plot the scan figure with a logarithmic axis instead
Sometimes it's easier to see how the data behaves with a logarithmic intensity axis instead. That is just done with a simple `logy=True` flag.

In [ ]:
scan.plot_monitor("He3H", logy=True)

#### Get the peak-to-background ratio
With a little more Python code, you can get a lot more information out of your data. This is just an example.

In [ ]:
import numpy as np   # numpy gives us array maths (max, argmax, mean, abs)

x, I, err = scan.get_monitor_data("He3H")   # x = QL values, I = intensity per point, err = error bar on I

# Print raw numbers so you can see where the peak and the flat background are
for xi, Ii, ei in zip(x, I, err):   # loop over the three arrays together, one scan point at a time
    print(f"QL = {xi:.3f}   I = {Ii:.4g} +/- {ei:.2g}")   # QL to 3 decimals, intensity to 4 significant digits, error to 2

peak = I.max()            # highest intensity in the whole scan = the Bragg peak height
peak_pos = x[I.argmax()]  # argmax gives the index of that highest point; x[...] looks up its QL value

# Background: average of the points far from the peak (adjust the 0.15 if the peak is even wider)
far = np.abs(x - peak_pos) > 0.15   # True for points more than 0.15 r.l.u. from the peak centre, False otherwise
background = I[far].mean()   # keep only the "far" points (boolean mask) and average them = background level

print(f"\nPeak at QL = {peak_pos:.3f}, intensity = {peak:.4g}")   # report where the peak is and how high
print(f"Background = {background:.4g}")                           # report the average background intensity
print(f"Peak / background = {peak / background:.1f}")             # the answer to the quiz: peak height divided by background

## Visualize instrument

In [ ]:
sim_interface.instrument.show_instrument(backend="pythreejs")